# 28 · Evaluation for revision 4 (T4 GPU, resumable; run after notebook 25, and again after notebook 27)
No training. Read-only for runs and data. Writes `results/p2_rev4/eval/`.

* **Frozen cohort** (notebook 25): every finished run in all run folders (the 190 study runs and the new runs) predicts every frozen crop;
  per-image Dice, IoU, HD95 and empty-prediction flag.
* **New runs only** (notebook 27): exact per-image scores on their CBIS-DDSM test fold and on the 301 DMID crops, and the code-replacement
  tests of notebook 24 (bank of the 240 training codes of the fold; const = exact mean; ten donors fixed by case identifier).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc, hashlib, random
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
FROZ_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
OUT      = f'{PROJECT}/results/p2_rev4'
ROOTS    = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val',
            'train4': f'{PROJECT}/results/p2_rev4_runs_train', 'val4': f'{PROJECT}/results/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
for d in [OUT] + list(ROOTS.values()): os.makedirs(d, exist_ok=True)
PROTO_FILE = f'{OUT}/PROTOCOL_rev4_prespecified.txt'
PROTO = """Paper 2 - revision 4 analyses, written before any of them was run (frozen cohort selected, crops made or runs trained).

A. Frozen evaluation on untouched patients. The pipeline is frozen: the 190 saved models, crop protocol, threshold 0.5 and metrics are
   unchanged. Cohort: CBIS-DDSM mass ROIs (same eligibility as the study: every mass ROI whose full-field image is available, no filtering)
   of patients NOT among the 208 study patients. Patients are shuffled once with Python random.Random(2026) and added whole, in that
   order, while the total stays at or below 400 ROIs (patients that would exceed 400 are skipped); selection stops at 400 ROIs.
   Crops: the study's randomized protocol (seed from the MD5 of the ROI identifier). Every saved model predicts every frozen crop; per
   crop and seed, Dice is averaged over the five fold models (as for DMID). Primary estimands: the nine model contrasts of Tables VI-VII
   under each schedule, crossed hierarchical bootstrap over patients and seeds (10,000 replicates, rng(0)), Holm within the nine
   contrasts of each schedule. Secondary: model levels, the two no-network baselines, empty-prediction rates, HD95.
B. Inference checks (no training). (i) Exact inference repeated three times within one session (same checkpoint, same inputs) and
   compared with the stored values, for the 60 circuit runs and their Dense(4) partners, on the CBIS-DDSM test fold and DMID.
   (ii) Initialization audit: for seeds 0-2 and fold 0, the global and patch-wise models (circuit, Dense(4), trigonometric, identity)
   are built with the training seed; the weights of the layers shared by all variants are compared, and the initial block input and
   output statistics and the gradient norms at the block are recorded on four training batches.
C. New training runs (training and evaluation exactly as in the study):
   C. Trigonometric control g_tr, p_tr: z = tanh(A cos((pi/2) x + phi)), A 4x4 Glorot uniform, phi uniform [0, 2 pi): 20 parameters,
       same slot as the circuit; seeds 0-2, folds 0-4, both schedules (60 runs).
   Primary estimands (CBIS-DDSM test folds): circuit - trigonometric at each placement under each schedule (4 contrasts, Holm over
   the four). Secondary: trigonometric - Dense(4) and trigonometric - identity at each placement; the same contrasts on DMID
   (exploratory) and on the frozen cohort; code-replacement drops of the trigonometric models.
   No further seeds and no further learning-rate schedule are trained.
All runs and all results are reported; none is discarded."""
if not os.path.exists(PROTO_FILE):
    open(PROTO_FILE, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print('protocol:', open(PROTO_FILE).read().splitlines()[-1])
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)


In [ ]:
# write the revision-4 module (trigonometric block) next to qseg2.py
open(f'{PROJECT}/code/qseg2_rev4.py', 'w').write('"""Revision-4 additions to qseg2 (Paper 2): a 20-parameter trigonometric block for the 4->4 slot.\n\n    z = tanh(A cos((pi/2) x + phi)),  A in R^{4x4} (Glorot uniform),  phi in R^4 (uniform on [0, 2 pi), as the circuit angles)\n\nThe angle encoding (pi/2) x is the one the circuit uses; like the circuit outputs, the block output is a trigonometric\nfunction of the angles. Global models apply it to the 4-value code, patch-wise models to every cell with one shared set of\nweights (as the patch-wise circuit). Models: g_tr, p_tr. Call patch() once after importing qseg2.\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\n\n\nclass TrigLayer(layers.Layer):\n    def __init__(self, n=4, **kw):\n        super().__init__(**kw)\n        self.n = n\n\n    def build(self, input_shape):\n        self.A = self.add_weight(name="A", shape=(self.n, self.n), initializer="glorot_uniform")\n        self.phi = self.add_weight(name="phi", shape=(self.n,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi))\n        super().build(input_shape)\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * tf.cast(x, tf.float32) + self.phi)\n        return tf.tanh(tf.einsum("...i,ji->...j", c, self.A))\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n    def get_config(self):\n        return {**super().get_config(), "n": self.n}\n\n\n_orig_block = qseg2._block\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    if kind == "tr":\n        return TrigLayer(n_qubits, name="trig_replacement")(z)\n    return _orig_block(z, kind, n_qubits, n_layers, patch)\n\n\ndef patch():\n    qseg2._block = _block\n    qseg2.MODELS2.setdefault("g_tr", "Global bottleneck, trigonometric 4->4 layer (20 parameters)")\n    qseg2.MODELS2.setdefault("p_tr", "Patch-wise bottleneck, trigonometric 4->4 layer (20 parameters)")\n    return qseg2\n')


In [ ]:
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
import qseg2_rev4; importlib.reload(qseg2_rev4); qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# 1 · Data
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
FM = pd.read_csv(f'{FROZ_DIR}/metadata.csv')
XF = np.stack([_png(f'{FROZ_DIR}/images/{r}.png', False) for r in FM.roi_id]); YF = np.stack([_png(f'{FROZ_DIR}/masks/{r}.png', True) for r in FM.roi_id])
meta = pd.read_csv(f'{DMID_DIR}/metadata.csv')
XD = np.stack([_png(f'{DMID_DIR}/images/{r}.png', False) for r in meta.roi_id]); YD = np.stack([_png(f'{DMID_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
print('frozen', XF.shape, '| DMID', XD.shape, '| CBIS', XC.shape)
for k in ('frozen', 'new_cbis_dmid', 'replace'): os.makedirs(f'{OUT}/eval/{k}', exist_ok=True)
REPL_G = tf.Variable(tf.zeros((B, 4)), trainable=False); REPL_P = tf.Variable(tf.zeros((B, 4, 4, 4)), trainable=False)
USE = tf.Variable(0.0, trainable=False)
if not getattr(qseg2.Sever, '_rev4', False):
    _sever_orig = qseg2.Sever.call
    def _sever_call(self, x):
        y = _sever_orig(self, x); r = REPL_G if len(x.shape) == 2 else REPL_P
        return tf.where(USE > 0.5, tf.cast(r[:tf.shape(x)[0]], y.dtype), y)
    qseg2.Sever.call = _sever_call; qseg2.Sever._rev4 = True
def load(d, name):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5'); return m
def forward(m):
    f = tf.function(lambda x: m(x, training=False), reduce_retracing=True)
    def run(X, repl=None):
        out = []
        for i in range(0, len(X), B):
            xb = X[i:i + B]
            if repl is None: USE.assign(0.0)
            else:
                r = np.zeros((B,) + repl.shape[1:], np.float32); r[:len(xb)] = repl[i:i + B]
                (REPL_G if repl.ndim == 2 else REPL_P).assign(r); USE.assign(1.0)
            out.append(f(tf.constant(xb)).numpy())
        USE.assign(0.0); return np.concatenate(out)
    return run
def codes_of(m, X):
    cm = tf.keras.Model(m.input, m.get_layer('sever').input)
    return np.concatenate([cm(tf.constant(X[i:i + B]), training=False).numpy() for i in range(0, len(X), B)]).astype(np.float32)
def donor(case_id, r, n):
    return int(np.random.default_rng(int(hashlib.md5(f'{case_id}|{r}'.encode()).hexdigest()[:8], 16)).integers(n))
def runs_of(root):
    return [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]

In [ ]:
# 2 · Frozen cohort: every finished run (resumable)
n = 0
for key, root in ROOTS.items():
    os.makedirs(f'{OUT}/eval/frozen/{key}', exist_ok=True)
    for d in runs_of(root):
        dst = f'{OUT}/eval/frozen/{key}/{os.path.basename(d)}.csv'
        if os.path.exists(dst): continue
        s = json.load(open(f'{d}/summary.json')); m = load(d, s['model'])
        P = forward(m)(XF); t = qseg2.per_image_scores(YF, P, cfg.threshold)
        t.insert(0, 'roi_id', FM.roi_id.values); t.insert(1, 'patient', FM.patient.values)
        t['empty'] = (P[..., 0] >= cfg.threshold).sum((1, 2)) == 0
        t['set'], t['schedule'], t['model'], t['fold'], t['seed'] = key, SCHED[key], s['model'], int(s['fold']), int(s['seed'])
        t.to_csv(dst, index=False); n += 1; del m; gc.collect()
        print(key, os.path.basename(d), 'frozen Dice', round(float(t.dice.mean()), 4), flush=True)
print('frozen: new evaluations', n)

In [ ]:
# 3 · New runs: CBIS-DDSM test fold, DMID and code replacement (as notebook 24, part A; resumable)
K = 10
for key in ('train4', 'val4'):
    os.makedirs(f'{OUT}/eval/replace/{key}', exist_ok=True)
    for d in runs_of(ROOTS[key]):
        dst = f'{OUT}/eval/replace/{key}/{os.path.basename(d)}.csv'
        if os.path.exists(dst): continue
        s = json.load(open(f'{d}/summary.json')); name, fold = s['model'], int(s['fold'])
        m = load(d, name); run = forward(m)
        tr, _, te = qseg2.fold_split(DFC, fold)
        order = np.argsort(DFC.roi_id.values[tr]); tr = tr[order]
        bank = codes_of(m, XC[tr]); const = bank.mean(0, keepdims=True)
        parts = []
        for ds, Xs, Ys, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
            P0 = run(Xs); t = qseg2.per_image_scores(Ys, P0, cfg.threshold); t.insert(0, 'roi_id', ids)
            t['empty'] = (P0[..., 0] >= cfg.threshold).sum((1, 2)) == 0
            Pc = run(Xs, np.repeat(const, len(Xs), 0)); t['dice_const'] = qseg2.per_image_scores(Ys, Pc, cfg.threshold).dice.values
            t['flip_const'] = ((Pc[..., 0] >= 0.5) != (P0[..., 0] >= 0.5)).mean((1, 2)); t['absdp_const'] = np.abs(Pc - P0)[..., 0].mean((1, 2))
            for r in range(K):
                idx = np.array([donor(c, r, len(bank)) for c in ids]); Ps = run(Xs, bank[idx])
                t[f'dice_swap{r}'] = qseg2.per_image_scores(Ys, Ps, cfg.threshold).dice.values
                if r == 0:
                    t['flip_swap'] = ((Ps[..., 0] >= 0.5) != (P0[..., 0] >= 0.5)).mean((1, 2)); t['absdp_swap'] = np.abs(Ps - P0)[..., 0].mean((1, 2))
            t.insert(0, 'dataset', ds); parts.append(t)
        out = pd.concat(parts, ignore_index=True)
        out['set'], out['schedule'], out['model'], out['fold'], out['seed'] = key, SCHED[key], name, fold, int(s['seed'])
        out['epochs_run'], out['best_epoch'], out['dice_training'] = s['epochs_run'], s['best_epoch'], s['dice_per_image']
        out.to_csv(dst, index=False); del m; gc.collect()
        print(key, os.path.basename(d), '| CBIS', round(float(out[out.dataset == 'CBIS'].dice.mean()), 4), 'vs training', round(s['dice_per_image'], 4),
              '| DMID', round(float(out[out.dataset == 'DMID'].dice.mean()), 4), flush=True)

In [ ]:
# 4 · Consolidate (full precision, gzip)
for k in ('frozen', 'replace'):
    fs = sorted(glob.glob(f'{OUT}/eval/{k}/*/*.csv'))
    if fs:
        a = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
        a.to_csv(f'{OUT}/rev4_{k}_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
        print(k, a.shape, '| runs', a.groupby(['set', 'model', 'fold', 'seed']).ngroups)
fz = pd.read_csv(f'{OUT}/rev4_frozen_per_image.csv.gz')
print(fz.groupby(['set', 'model']).dice.mean().unstack(0).round(4).to_string())

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()